# Sweep Line

The last pattern. Imagine a vertical line sweeping left-to-right across a number line, stopping at each **event** — an interval's start or end, a point's coordinate. Sort the events once, then process them in order while maintaining a small **active state**, and a whole class of interval and geometry problems collapses to a single O(n log n) pass.

> signal → template → worked examples → toolkit

**Contents**

1. **The signal**
2. **Merge overlapping intervals**
3. **Maximum concurrency** — the event sweep
4. **When to use & cheat-sheet**

## 1. The signal — events sorted along an axis

Reach for a sweep line when the problem is about **intervals or points on a line** (times, coordinates, ranges) and asks about their **overlap or coverage**:

- "merge overlapping intervals," "do any intervals overlap?"
- "maximum number overlapping at once" ("minimum meeting rooms")
- "total covered length," "the skyline," "count points falling in ranges."

The pattern: turn the inputs into **events** (a start, an end, a query point), **sort** them along the axis, then sweep through in order, updating a small piece of running **state** (a count, an active set, the last interval) at each event. Sorting is the O(n log n); the sweep itself is O(n).

## 2. Merge overlapping intervals

The simplest sweep. Sort intervals by start; walk them keeping the last merged run. If the next interval starts **before the current run ends**, they overlap — extend the end; otherwise begin a fresh run:

In [1]:
def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):         # sweep left to right by start
        if merged and start <= merged[-1][1]:    # overlaps the current run?
            merged[-1][1] = max(merged[-1][1], end)   # extend it
        else:
            merged.append([start, end])          # a gap -> begin a new run
    return merged

print("merge([[1,3],[2,6],[8,10],[15,18]]):", merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))


merge([[1,3],[2,6],[8,10],[15,18]]): [[1, 6], [8, 10], [15, 18]]


"Interval list intersection," "insert interval," and "can attend all meetings?" are the same sort-by-start sweep with a different per-step update.

## 3. Maximum concurrency — the event sweep

For "how many intervals overlap at the busiest moment?" (the **minimum meeting rooms** problem), split each interval into two **events**: `+1` at its start, `-1` at its end. Sort all events by time and sweep a running counter — its peak is the answer. The crucial tie-break: at equal times, process **ends before starts** (`-1` sorts before `+1`), so a meeting that ends exactly when another begins can share the room:

In [2]:
def min_meeting_rooms(intervals):
    events = []
    for start, end in intervals:
        events.append((start, +1))               # a meeting begins -> need a room
        events.append((end, -1))                 # a meeting ends   -> free a room
    events.sort()                                # ends sort before starts at equal time (-1 < +1)
    active = best = 0
    for _, delta in events:
        active += delta
        best = max(best, active)
    return best

print("min_meeting_rooms([[0,30],[5,10],[15,20]]):", min_meeting_rooms([[0, 30], [5, 10], [15, 20]]))
print("min_meeting_rooms([[1,5],[2,6],[3,7]])    :", min_meeting_rooms([[1, 5], [2, 6], [3, 7]]))


min_meeting_rooms([[0,30],[5,10],[15,20]]): 2
min_meeting_rooms([[1,5],[2,6],[3,7]])    : 3


The same answer comes from a **heap of end times** — push each interval's end as you sweep by start, pop ends that finish before the current start, and the heap's size *is* the concurrency. (That's the top-K / heap pattern showing up again.)

In [3]:
import heapq

def min_rooms_heap(intervals):
    heap, best = [], 0                            # heap holds end times of active meetings
    for start, end in sorted(intervals):
        while heap and heap[0] <= start:
            heapq.heappop(heap)                  # a room freed at/before this start
        heapq.heappush(heap, end)
        best = max(best, len(heap))
    return best

print("min_rooms_heap([[0,30],[5,10],[15,20]]):", min_rooms_heap([[0, 30], [5, 10], [15, 20]]))


min_rooms_heap([[0,30],[5,10],[15,20]]): 2


## 4. When to use & cheat-sheet

| The problem is about… | Sweep with… |
|---|---|
| merge / insert / intersect intervals | sort by start, track the last run |
| max overlap / min rooms | ±1 events, or a heap of end times |
| total covered length / union | sort events, accumulate while active > 0 |
| skyline / max height | events + a max-heap of heights |
| points-in-ranges, closest pair | sweep + an ordered active set |

**Python toolkit:** `sorted(..., key=...)` builds the event order (the `key` pins the tie-break); `heapq` maintains the active set's min/max; `collections.Counter` or a `SortedList` (`sortedcontainers`) tracks active multiplicities. For huge coordinate ranges, **compress** them first (coordinate-compression notebook).

**The tie-break is everything:** whether `+1` or `-1` wins at equal coordinates decides off-by-one behavior (do touching intervals share a room?). Pin it down with an explicit sort key.

| Task | Approach | Time |
|---|---|:---:|
| Merge / intersect intervals | sort by start, one pass | O(n log n) |
| Max overlap / min rooms | ±1 events / end-time heap | O(n log n) |
| Covered length / union | sort events, accumulate | O(n log n) |

---

**That's the patterns tier — and the repo.** Sweep line is a fitting close: it reuses `sorted` (sorting notebook), `heapq` (heaps), coordinate compression, and the top-K heap idea — the same recurring move of *impose an order, then make one clean pass*. Across all four tiers — foundations, data structures, algorithms, patterns — that's been the throughline: the right structure or ordering turns a brute-force search into a single efficient sweep.